# Instaloader 실전 튜토리얼

이 노트북은 Instaloader를 다양한 방법으로 테스트하고, 수집한 데이터를 분석하는 방법을 다룸.

## 목차
1. [환경 설정](#1-환경-설정)
2. [CLI 기반 테스트](#2-cli-기반-테스트)
3. [Python API 기본 사용법](#3-python-api-기본-사용법)
4. [고급 활용 - 필터링과 커스터마이징](#4-고급-활용---필터링과-커스터마이징)
5. [데이터 분석 - JSON/XZ 파일 읽기](#5-데이터-분석---jsonxz-파일-읽기)
6. [종합 데이터 분석](#6-종합-데이터-분석)
7. [실전 팁과 트러블슈팅](#7-실전-팁과-트러블슈팅)

---

## 1. 환경 설정

In [1]:
# 필요한 라이브러리 설치 (처음 한 번만 실행)
!pip install instaloader pandas matplotlib seaborn

In [2]:
# 기본 라이브러리 임포트
import instaloader
import json
import lzma
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from pathlib import Path
from collections import defaultdict

# 한글 폰트 설정 (matplotlib)
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"Instaloader 버전: {instaloader.__version__}")
print("환경 설정 완료!")

Instaloader 버전: 4.15
환경 설정 완료!


In [3]:
# 작업 디렉토리 설정
WORK_DIR = Path("../instaloader_data")
WORK_DIR.mkdir(exist_ok=True)

print(f"작업 디렉토리: {WORK_DIR.absolute()}")

작업 디렉토리: /workspace/instaloader/my_docs/../instaloader_data


---

## 2. CLI 기반 테스트

주피터 노트북에서 `!` 또는 `%%bash`를 사용하여 CLI 명령어를 실행할 수 있음.

### 2.1 도움말 확인

In [ ]:
# Instaloader 버전 및 도움말
!instaloader --version

In [ ]:
# 전체 옵션 보기 (많아서 일부만 보기)
!instaloader --help | head -50

### 2.2 익명 다운로드 테스트 (로그인 없이)

공개 프로필의 최근 게시글 3개만 다운로드

In [ ]:
%%bash
cd instaloader_data

# 공개 프로필 테스트 (예: instagram 공식 계정)
# --count 3: 최근 3개만
# --no-video-thumbnails: 비디오 썸네일 제외
# --no-metadata-json: JSON 메타데이터 제외 (빠른 테스트용)
instaloader --count 3 --no-video-thumbnails instagram

echo "\n다운로드 완료! 파일 목록:"
ls -lh instagram/ | head -10

### 2.3 프로필 메타데이터만 가져오기

In [ ]:
%%bash
cd instaloader_data

# 프로필 정보만 가져오기 (게시글 다운로드 안 함)
instaloader --no-posts --no-profile-pic instagram

# JSON 파일 확인
echo "\n프로필 JSON:"
cat instagram/instagram_*.json.xz | xz -d | jq '{username, full_name, biography, followers: .edge_followed_by.count, following: .edge_follow.count, posts: .edge_owner_to_timeline_media.count}'

### 2.4 해시태그 다운로드

In [ ]:
%%bash
cd instaloader_data

# 해시태그 최근 5개 게시글
instaloader --count 5 "#python"

echo "\n다운로드된 파일:"
ls -lh "#python/" | head -10

### 2.5 특정 게시글 다운로드 (shortcode 사용)

In [ ]:
%%bash
cd instaloader_data

# Instagram URL: https://www.instagram.com/p/ABC123xyz/
# shortcode: ABC123xyz
instaloader -- -ABC123xyz

# 실제 예시 (Instagram 공식 계정의 최근 게시글 하나)
# 먼저 shortcode를 알아야 함 (다음 섹션에서 Python API로 확인 가능)

---

## 3. Python API 기본 사용법

### 3.1 Instaloader 인스턴스 생성

In [ ]:
# Instaloader 인스턴스 생성 (익명 모드)
L = instaloader.Instaloader(
    dirname_pattern=str(WORK_DIR / "{target}"),  # 저장 경로
    download_pictures=True,          # 이미지 다운로드
    download_videos=False,           # 비디오 제외 (빠른 테스트)
    download_video_thumbnails=False, # 비디오 썸네일 제외
    download_geotags=False,          # 위치 정보 제외
    download_comments=False,         # 댓글 제외 (익명은 불가능)
    save_metadata=True,              # JSON 메타데이터 저장
    compress_json=True,              # JSON 압축 (.xz)
    post_metadata_txt_pattern="",   # txt 파일 생성 안 함
    max_connection_attempts=3,       # 최대 재시도 횟수
)

print(f"Instaloader 준비 완료!")
print(f"로그인 상태: {L.context.is_logged_in}")
print(f"저장 경로: {WORK_DIR}")

### 3.2 프로필 정보 가져오기

In [ ]:
# 프로필 객체 생성
profile = instaloader.Profile.from_username(L.context, "instagram")

# 기본 정보 출력
print(f"사용자명: {profile.username}")
print(f"실명: {profile.full_name}")
print(f"User ID: {profile.userid}")
print(f"팔로워: {profile.followers:,}명")
print(f"팔로잉: {profile.followees:,}명")
print(f"게시글 수: {profile.mediacount:,}개")
print(f"비공개 계정: {profile.is_private}")
print(f"인증 계정: {profile.is_verified}")
print(f"비즈니스 계정: {profile.is_business_account}")
print(f"\n바이오:\n{profile.biography}")
print(f"\n외부 링크: {profile.external_url}")

### 3.3 최근 게시글 미리보기 (다운로드 안 함)

In [ ]:
# 최근 5개 게시글 정보만 출력
posts = profile.get_posts()

for i, post in enumerate(posts, 1):
    print(f"\n=== 게시글 {i} ===")
    print(f"Shortcode: {post.shortcode}")
    print(f"URL: https://www.instagram.com/p/{post.shortcode}/")
    print(f"날짜: {post.date_local}")
    print(f"타입: {post.typename} ({'동영상' if post.is_video else '이미지'})")
    print(f"좋아요: {post.likes:,}")
    print(f"댓글: {post.comments:,}")
    
    # 캡션 (앞부분만)
    caption = post.caption or "(캡션 없음)"
    print(f"캡션: {caption[:100]}..." if len(caption) > 100 else f"캡션: {caption}")
    
    if i >= 5:  # 5개만
        break

### 3.4 게시글 다운로드

In [ ]:
# 특정 개수만 다운로드
profile = instaloader.Profile.from_username(L.context, "instagram")
posts = profile.get_posts()

download_count = 0
max_download = 3  # 최대 3개

for post in posts:
    if download_count >= max_download:
        break
    
    try:
        print(f"다운로드 중: {post.shortcode} ({post.date_local})")
        L.download_post(post, target="instagram")
        download_count += 1
        print(f"✓ 완료 ({download_count}/{max_download})")
    except Exception as e:
        print(f"✗ 에러: {e}")

print(f"\n총 {download_count}개 다운로드 완료!")

### 3.5 특정 게시글만 다운로드 (shortcode 사용)

In [ ]:
# shortcode로 특정 게시글 다운로드
# 예: https://www.instagram.com/p/ABC123xyz/ → shortcode = "ABC123xyz"

shortcode = "CxOWiQNP2MV"  # 예시 (실제로는 위에서 얻은 shortcode 사용)

try:
    post = instaloader.Post.from_shortcode(L.context, shortcode)
    print(f"게시글 정보:")
    print(f"  작성자: {post.owner_username}")
    print(f"  날짜: {post.date_local}")
    print(f"  좋아요: {post.likes:,}")
    print(f"  댓글: {post.comments:,}")
    
    L.download_post(post, target="single_post")
    print(f"\n다운로드 완료!")
except Exception as e:
    print(f"에러: {e}")

---

## 4. 고급 활용 - 필터링과 커스터마이징

### 4.1 날짜 필터링

In [ ]:
# 특정 기간의 게시글만 다운로드
from datetime import datetime, timedelta

profile = instaloader.Profile.from_username(L.context, "instagram")

# 2024년 1월 1일 ~ 2024년 12월 31일
start_date = datetime(2024, 1, 1)
end_date = datetime(2024, 12, 31)

print(f"기간: {start_date.date()} ~ {end_date.date()}")
print("\n해당 기간 게시글:")

for post in profile.get_posts():
    post_date = post.date_local.replace(tzinfo=None)  # timezone 제거
    
    # 시작일 이전이면 중단 (최신순이니까)
    if post_date < start_date:
        print(f"\n시작일 이전 도달, 중단")
        break
    
    # 범위 내면 출력
    if start_date <= post_date <= end_date:
        print(f"  {post.date_local.date()} - {post.shortcode} - 좋아요 {post.likes:,}")
        # L.download_post(post, target="instagram_2024")  # 실제 다운로드

### 4.2 조건부 다운로드 (좋아요 수, 타입 등)

In [ ]:
# 좋아요 10만 이상, 이미지만 다운로드
profile = instaloader.Profile.from_username(L.context, "instagram")

min_likes = 100000
count = 0
max_count = 5

print(f"조건: 좋아요 {min_likes:,}개 이상, 이미지만")
print("\n조건 만족 게시글:")

for post in profile.get_posts():
    if count >= max_count:
        break
    
    # 조건 체크
    if post.likes >= min_likes and not post.is_video:
        print(f"  {post.shortcode} - 좋아요 {post.likes:,}")
        # L.download_post(post, target="popular_images")
        count += 1

print(f"\n총 {count}개 발견")

### 4.3 커스텀 필터 함수 사용

In [ ]:
# 복잡한 조건의 필터 함수
def my_filter(post):
    """
    조건:
    1. 좋아요 5만 이상
    2. 댓글 1천 이상
    3. 최근 30일 이내
    4. 이미지 또는 슬라이드
    """
    # 날짜 체크
    days_ago = (datetime.now() - post.date_local.replace(tzinfo=None)).days
    
    return (
        post.likes >= 50000 and
        post.comments >= 1000 and
        days_ago <= 30 and
        post.typename in ["GraphImage", "GraphSidecar"]
    )

# 필터 적용
profile = instaloader.Profile.from_username(L.context, "instagram")

print("커스텀 필터 적용 중...\n")
matched = []

for post in profile.get_posts():
    if my_filter(post):
        matched.append(post)
        print(f"✓ {post.shortcode} - 좋아요 {post.likes:,}, 댓글 {post.comments:,}")
    
    # 최근 50개만 체크
    if len(matched) + (profile.get_posts().count - 1) >= 50:
        break

print(f"\n조건 만족: {len(matched)}개")

### 4.4 해시태그 수집

In [ ]:
# 해시태그 게시글 수집
hashtag = instaloader.Hashtag.from_name(L.context, "python")

print(f"해시태그: #{hashtag.name}")
print(f"게시글 수: {hashtag.mediacount:,}개\n")

print("최근 게시글:")
for i, post in enumerate(hashtag.get_posts(), 1):
    print(f"{i}. @{post.owner_username} - {post.date_local.date()} - 좋아요 {post.likes:,}")
    
    if i >= 10:  # 10개만
        break

---

## 5. 데이터 분석 - JSON/XZ 파일 읽기

### 5.1 단일 JSON.XZ 파일 읽기

In [ ]:
def read_json_xz(file_path):
    """
    압축된 JSON 파일 (.json.xz) 읽기
    
    Args:
        file_path: .json.xz 파일 경로
    
    Returns:
        dict: JSON 데이터
    """
    with lzma.open(file_path, 'rt', encoding='utf-8') as f:
        return json.load(f)

# 예시: 다운로드한 게시글 JSON 읽기
json_files = list(WORK_DIR.glob("instagram/*_UTC.json.xz"))

if json_files:
    first_file = json_files[0]
    print(f"파일: {first_file.name}")
    
    data = read_json_xz(first_file)
    
    print(f"\n주요 정보:")
    print(f"  Shortcode: {data.get('shortcode')}")
    print(f"  타입: {data.get('__typename')}")
    print(f"  좋아요: {data.get('edge_media_preview_like', {}).get('count', 0):,}")
    print(f"  댓글: {data.get('edge_media_to_parent_comment', {}).get('count', 0):,}")
    print(f"  작성일: {datetime.fromtimestamp(data.get('taken_at_timestamp', 0))}")
    
    # 전체 구조 보기 (일부만)
    print(f"\nJSON 구조 (최상위 키):")
    for key in list(data.keys())[:10]:
        print(f"  - {key}")
else:
    print("JSON 파일이 없습니다. 먼저 게시글을 다운로드하세요.")

### 5.2 프로필 JSON 읽기

In [ ]:
# 프로필 JSON 파일 찾기 (패턴: {username}_{userid}.json.xz)
profile_json_files = list(WORK_DIR.glob("instagram/instagram_*.json.xz"))

# 게시글 JSON 제외 (날짜가 포함된 파일 제외)
profile_json_files = [
    f for f in profile_json_files 
    if "_UTC" not in f.name
]

if profile_json_files:
    profile_file = profile_json_files[0]
    print(f"프로필 파일: {profile_file.name}")
    
    profile_data = read_json_xz(profile_file)
    
    print(f"\n프로필 정보:")
    print(f"  사용자명: {profile_data.get('username')}")
    print(f"  실명: {profile_data.get('full_name')}")
    print(f"  User ID: {profile_data.get('id')}")
    print(f"  팔로워: {profile_data.get('edge_followed_by', {}).get('count', 0):,}")
    print(f"  팔로잉: {profile_data.get('edge_follow', {}).get('count', 0):,}")
    print(f"  게시글: {profile_data.get('edge_owner_to_timeline_media', {}).get('count', 0):,}")
    print(f"  비공개: {profile_data.get('is_private')}")
    print(f"  인증: {profile_data.get('is_verified')}")
    print(f"\n바이오:\n{profile_data.get('biography')}")
else:
    print("프로필 JSON 파일이 없습니다.")

### 5.3 여러 게시글 JSON을 DataFrame으로 변환

In [ ]:
def posts_to_dataframe(json_dir):
    """
    디렉토리 내 모든 게시글 JSON.XZ 파일을 읽어서 DataFrame으로 변환
    
    Args:
        json_dir: JSON 파일들이 있는 디렉토리 경로
    
    Returns:
        pd.DataFrame: 게시글 데이터
    """
    json_files = list(Path(json_dir).glob("*_UTC.json.xz"))
    
    if not json_files:
        print(f"경고: {json_dir}에 게시글 JSON 파일이 없습니다.")
        return pd.DataFrame()
    
    posts_data = []
    
    for json_file in json_files:
        try:
            data = read_json_xz(json_file)
            
            # 필요한 정보만 추출
            post_info = {
                'shortcode': data.get('shortcode'),
                'typename': data.get('__typename'),
                'is_video': data.get('is_video', False),
                'date': datetime.fromtimestamp(data.get('taken_at_timestamp', 0)),
                'likes': data.get('edge_media_preview_like', {}).get('count', 0),
                'comments': data.get('edge_media_to_parent_comment', {}).get('count', 0),
                'caption': data.get('edge_media_to_caption', {}).get('edges', [{}])[0].get('node', {}).get('text', ''),
                'location': data.get('location', {}).get('name', None) if data.get('location') else None,
                'owner_username': data.get('owner', {}).get('username', ''),
                'owner_id': data.get('owner', {}).get('id', ''),
                'url': f"https://www.instagram.com/p/{data.get('shortcode')}/",
            }
            
            # 해시태그 추출
            caption = post_info['caption']
            hashtags = [word[1:] for word in caption.split() if word.startswith('#')]
            post_info['hashtags'] = hashtags
            post_info['hashtag_count'] = len(hashtags)
            
            posts_data.append(post_info)
            
        except Exception as e:
            print(f"에러 ({json_file.name}): {e}")
            continue
    
    df = pd.DataFrame(posts_data)
    
    # 날짜순 정렬
    if not df.empty:
        df = df.sort_values('date', ascending=False).reset_index(drop=True)
    
    return df

# 테스트
df_posts = posts_to_dataframe(WORK_DIR / "instagram")

if not df_posts.empty:
    print(f"총 {len(df_posts)}개 게시글 로드 완료\n")
    print(df_posts[['date', 'typename', 'likes', 'comments', 'hashtag_count']].head(10))
else:
    print("DataFrame이 비어있습니다.")

---

## 6. 종합 데이터 분석

### 6.1 여러 프로필 데이터 병합

In [ ]:
def merge_multiple_profiles(base_dir):
    """
    여러 프로필의 게시글 JSON을 하나의 DataFrame으로 병합
    
    Args:
        base_dir: 프로필 디렉토리들이 있는 상위 디렉토리
    
    Returns:
        pd.DataFrame: 병합된 게시글 데이터
    """
    base_path = Path(base_dir)
    all_posts = []
    
    # 모든 하위 디렉토리 탐색
    for profile_dir in base_path.iterdir():
        if profile_dir.is_dir() and not profile_dir.name.startswith('.'):
            print(f"로딩 중: {profile_dir.name}...")
            df = posts_to_dataframe(profile_dir)
            
            if not df.empty:
                df['profile_name'] = profile_dir.name  # 프로필명 추가
                all_posts.append(df)
    
    if all_posts:
        merged_df = pd.concat(all_posts, ignore_index=True)
        print(f"\n총 {len(merged_df)}개 게시글 로드 완료")
        return merged_df
    else:
        print("게시글이 없습니다.")
        return pd.DataFrame()

# 테스트
df_all = merge_multiple_profiles(WORK_DIR)

if not df_all.empty:
    print(f"\n프로필별 게시글 수:")
    print(df_all['profile_name'].value_counts())

### 6.2 기본 통계

In [ ]:
if not df_posts.empty:
    print("=== 기본 통계 ===")
    print(f"\n총 게시글 수: {len(df_posts)}개")
    print(f"기간: {df_posts['date'].min().date()} ~ {df_posts['date'].max().date()}")
    print(f"\n타입 분포:")
    print(df_posts['typename'].value_counts())
    print(f"\n동영상 비율: {df_posts['is_video'].mean():.1%}")
    
    print(f"\n=== 인게이지먼트 통계 ===")
    print(df_posts[['likes', 'comments']].describe())
    
    print(f"\n=== 해시태그 통계 ===")
    print(df_posts['hashtag_count'].describe())
    
    # 인기 게시글 TOP 5
    print(f"\n=== TOP 5 (좋아요 기준) ===")
    top5 = df_posts.nlargest(5, 'likes')[['date', 'likes', 'comments', 'url']]
    for idx, row in top5.iterrows():
        print(f"{row['date'].date()} - 좋아요 {row['likes']:,} - {row['url']}")

### 6.3 시각화

In [ ]:
if not df_posts.empty and len(df_posts) > 1:
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # 1. 시간대별 게시글 수
    df_posts.set_index('date').resample('D').size().plot(ax=axes[0, 0], title='Daily Posts')
    axes[0, 0].set_ylabel('Number of Posts')
    
    # 2. 좋아요 분포
    df_posts['likes'].hist(bins=30, ax=axes[0, 1])
    axes[0, 1].set_title('Likes Distribution')
    axes[0, 1].set_xlabel('Likes')
    axes[0, 1].set_ylabel('Frequency')
    
    # 3. 좋아요 vs 댓글
    axes[1, 0].scatter(df_posts['likes'], df_posts['comments'], alpha=0.5)
    axes[1, 0].set_title('Likes vs Comments')
    axes[1, 0].set_xlabel('Likes')
    axes[1, 0].set_ylabel('Comments')
    
    # 4. 타입별 평균 좋아요
    df_posts.groupby('typename')['likes'].mean().plot(kind='bar', ax=axes[1, 1])
    axes[1, 1].set_title('Average Likes by Type')
    axes[1, 1].set_ylabel('Average Likes')
    axes[1, 1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()
else:
    print("시각화를 위한 충분한 데이터가 없습니다.")

### 6.4 해시태그 분석

In [ ]:
if not df_posts.empty:
    # 모든 해시태그 추출
    all_hashtags = []
    for hashtags in df_posts['hashtags']:
        all_hashtags.extend(hashtags)
    
    if all_hashtags:
        # 빈도수 계산
        from collections import Counter
        hashtag_counts = Counter(all_hashtags)
        
        print("=== 인기 해시태그 TOP 20 ===")
        for hashtag, count in hashtag_counts.most_common(20):
            print(f"#{hashtag}: {count}회")
        
        # 시각화
        top_hashtags = dict(hashtag_counts.most_common(15))
        
        plt.figure(figsize=(12, 6))
        plt.barh(list(top_hashtags.keys()), list(top_hashtags.values()))
        plt.xlabel('Frequency')
        plt.title('Top 15 Hashtags')
        plt.gca().invert_yaxis()
        plt.tight_layout()
        plt.show()
    else:
        print("해시태그가 없습니다.")

### 6.5 시간대 분석

In [ ]:
if not df_posts.empty:
    # 시간대별 분석
    df_posts['hour'] = df_posts['date'].dt.hour
    df_posts['day_of_week'] = df_posts['date'].dt.day_name()
    df_posts['month'] = df_posts['date'].dt.month
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # 시간대별 게시글 수
    df_posts['hour'].value_counts().sort_index().plot(kind='bar', ax=axes[0])
    axes[0].set_title('Posts by Hour')
    axes[0].set_xlabel('Hour')
    axes[0].set_ylabel('Number of Posts')
    
    # 요일별 게시글 수
    day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    df_posts['day_of_week'].value_counts().reindex(day_order).plot(kind='bar', ax=axes[1])
    axes[1].set_title('Posts by Day of Week')
    axes[1].set_xlabel('Day')
    axes[1].tick_params(axis='x', rotation=45)
    
    # 월별 게시글 수
    df_posts['month'].value_counts().sort_index().plot(kind='bar', ax=axes[2])
    axes[2].set_title('Posts by Month')
    axes[2].set_xlabel('Month')
    
    plt.tight_layout()
    plt.show()
    
    # 통계 출력
    print("\n=== 가장 활발한 시간대 ===")
    print(f"시간: {df_posts['hour'].mode().values[0]}시")
    print(f"요일: {df_posts['day_of_week'].mode().values[0]}")
    print(f"월: {df_posts['month'].mode().values[0]}월")

### 6.6 DataFrame을 CSV로 저장

In [ ]:
if not df_posts.empty:
    # CSV 저장 (해시태그는 문자열로 변환)
    df_export = df_posts.copy()
    df_export['hashtags'] = df_export['hashtags'].apply(lambda x: ','.join(x) if x else '')
    
    csv_path = WORK_DIR / "posts_analysis.csv"
    df_export.to_csv(csv_path, index=False, encoding='utf-8-sig')
    print(f"CSV 저장 완료: {csv_path}")
    
    # Excel로도 저장
    xlsx_path = WORK_DIR / "posts_analysis.xlsx"
    df_export.to_excel(xlsx_path, index=False, engine='openpyxl')
    print(f"Excel 저장 완료: {xlsx_path}")

---

## 7. 실전 팁과 트러블슈팅

### 7.1 로그인 (세션 저장/로드)

In [ ]:
# 로그인 (처음 한 번만)
# 주의: 실제 계정 정보 입력 필요!

USERNAME = "your_username"  # 여기에 Instagram 아이디 입력
PASSWORD = "your_password"  # 여기에 비밀번호 입력

# 로그인
# L.login(USERNAME, PASSWORD)

# 세션 저장 (다음부터는 로그인 안 해도 됨)
# L.save_session_to_file()

print("⚠️ 보안을 위해 실제 코드는 주석 처리됨")
print("로그인하려면 위 코드의 주석을 제거하고 계정 정보를 입력하세요.")

In [ ]:
# 저장된 세션 로드
USERNAME = "your_username"

try:
    # L.load_session_from_file(USERNAME)
    # print(f"로그인 성공: {L.context.username}")
    print("⚠️ 세션 로드 코드는 주석 처리됨")
except FileNotFoundError:
    print("세션 파일이 없습니다. 먼저 로그인하세요.")

### 7.2 Rate Limit 대응

In [ ]:
# Rate Limit을 고려한 다운로드
import time
import random

def download_with_delay(profile_name, max_posts=10):
    """
    대기 시간을 두고 다운로드
    """
    profile = instaloader.Profile.from_username(L.context, profile_name)
    
    count = 0
    for post in profile.get_posts():
        if count >= max_posts:
            break
        
        try:
            print(f"다운로드 중: {post.shortcode}")
            L.download_post(post, target=profile_name)
            count += 1
            
            # 랜덤 대기 (2~5초)
            delay = random.uniform(2, 5)
            print(f"대기 중: {delay:.1f}초...")
            time.sleep(delay)
            
        except Exception as e:
            print(f"에러: {e}")
            # 429 에러면 10분 대기
            if "429" in str(e):
                print("Rate Limit! 10분 대기...")
                time.sleep(600)
    
    print(f"\n완료: {count}개 다운로드")

# 사용 예시
# download_with_delay("instagram", max_posts=5)

### 7.3 에러 핸들링

In [ ]:
# 안전한 다운로드 함수
def safe_download(profile_name, max_posts=10):
    """
    에러가 나도 계속 진행하는 다운로드
    """
    try:
        profile = instaloader.Profile.from_username(L.context, profile_name)
    except instaloader.exceptions.ProfileNotExistsException:
        print(f"에러: '{profile_name}' 프로필이 존재하지 않습니다.")
        return
    except instaloader.exceptions.LoginRequiredException:
        print(f"에러: '{profile_name}'은 비공개 계정입니다. 로그인이 필요합니다.")
        return
    
    success_count = 0
    error_count = 0
    errors = []
    
    for i, post in enumerate(profile.get_posts(), 1):
        if success_count >= max_posts:
            break
        
        try:
            print(f"[{i}] {post.shortcode}...", end=" ")
            L.download_post(post, target=profile_name)
            print("✓")
            success_count += 1
            
        except instaloader.exceptions.ConnectionException as e:
            print(f"✗ (연결 오류)")
            error_count += 1
            errors.append(f"{post.shortcode}: {str(e)}")
            time.sleep(5)  # 5초 대기 후 계속
            
        except Exception as e:
            print(f"✗ (기타 오류)")
            error_count += 1
            errors.append(f"{post.shortcode}: {str(e)}")
    
    print(f"\n완료! 성공: {success_count}, 실패: {error_count}")
    
    if errors:
        print("\n에러 목록:")
        for error in errors[:5]:  # 최대 5개만
            print(f"  - {error}")

# 사용 예시
# safe_download("instagram", max_posts=10)

### 7.4 진행 상황 추적

In [ ]:
# tqdm으로 진행 바 표시
try:
    from tqdm.notebook import tqdm
except ImportError:
    !pip install tqdm
    from tqdm.notebook import tqdm

def download_with_progress(profile_name, max_posts=20):
    """
    진행 바와 함께 다운로드
    """
    profile = instaloader.Profile.from_username(L.context, profile_name)
    posts = profile.get_posts()
    
    with tqdm(total=max_posts, desc="다운로드") as pbar:
        count = 0
        for post in posts:
            if count >= max_posts:
                break
            
            try:
                L.download_post(post, target=profile_name)
                pbar.set_postfix({"shortcode": post.shortcode})
                count += 1
                pbar.update(1)
            except Exception as e:
                pbar.set_postfix({"error": str(e)[:30]})
                continue

# 사용 예시
# download_with_progress("instagram", max_posts=10)

---

## 마무리

이 노트북을 통해:
1. ✅ CLI와 Python API 모두 테스트 가능
2. ✅ 다양한 조건으로 필터링 및 다운로드
3. ✅ JSON.XZ 파일을 DataFrame으로 변환
4. ✅ 여러 프로필 데이터 병합 및 분석
5. ✅ 시각화 및 통계 분석

### 다음 단계
- 로그인하여 스토리, 댓글 등 추가 데이터 수집
- 더 복잡한 필터와 조건 적용
- 수집한 데이터로 머신러닝 모델 학습
- 정기적인 데이터 수집 자동화